## build_fact_fred
Rebuilds `gold.fact_fred_national_monthly` from `silver.fact_fred_series` (long): pivots the 10 series to wide columns on a **monthly spine** (from `dim_date`, clipped to the FRED data range) and **forward-fills** each series as-of each month-end (the latest observation at or before that month) — this absorbs the 2025-10 BLS release gap (§4.2). National grain, PK `date_key`, no geo. Source values carried at native precision. Full rebuild via `INSERT OVERWRITE`. Spec: `gold_layer_design.md` §3.7 / §4.2.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
from pyspark.sql.window import Window

STEP_SEQUENCE = 7                       # position owned by the orchestrator (G4)
SOURCE_TABLE  = f"{SILVER}.fact_fred_series"
TARGET_TABLE  = f"{GOLD}.fact_fred_national_monthly"
DIM_DATE      = f"{GOLD}.dim_date"

# series_id -> Gold wide column, IN G0 COLUMN ORDER (gold_ddl.py). dict preserves order.
SERIES_MAP = {
    "MORTGAGE30US":  "mortgage_rate_30yr_pct",
    "MORTGAGE15US":  "mortgage_rate_15yr_pct",
    "FIXHAI":        "housing_affordability_index",
    "MSPUS":         "median_sales_price_usd",
    "UNRATE":        "unemployment_rate_pct",
    "MEHOINUSA672N": "real_median_household_income_usd",
    "ACTLISCOUUS":   "active_listing_count",
    "CSUSHPISA":     "case_shiller_hpi_sa",
    "CSUSHPINSA":    "case_shiller_hpi_nsa",
    "CPIAUCSL":      "cpi_all_urban_sa",
}

In [ ]:
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "gold",
    target_table    = TARGET_TABLE,
)
print(f"build_fact_fred: step_log_id={step.step_log_id}")

In [ ]:
# Long -> wide, forward-filled. (1) last obs per (series, month-end); (2) monthly spine from
# dim_date clipped to the FRED range; (3) as-of: each spine month takes the latest obs at or before
# it (carries values across gap months); (4) pivot series -> the 10 Gold columns.
try:
    src = spark.table(SOURCE_TABLE)
    rows_read = src.count()

    obs = (src.where(F.col("series_id").isin(list(SERIES_MAP.keys())))
              .select("series_id", F.last_day("observation_date").alias("month_end"),
                      "observation_date", "value"))
    last_w = Window.partitionBy("series_id", "month_end").orderBy(F.col("observation_date").desc())
    obs_last = (obs.withColumn("rn", F.row_number().over(last_w)).where(F.col("rn") == 1)
                   .select("series_id", "month_end", "value"))

    bounds = obs_last.agg(F.min("month_end").alias("lo"), F.max("month_end").alias("hi")).first()
    # dim_date is DAILY-grain (one row per calendar day, for the Power BI Date Table); filter to
    # month-end rows so this stays a MONTHLY spine. Without is_month_end the spine — and this whole
    # fact — would silently go daily, and the post_count==spine_count assert below would NOT catch
    # it (both sides would be the daily count). The filter is load-bearing.
    spine = (spark.table(DIM_DATE).where(F.col("is_month_end"))
               .select("date_key", F.col("full_date").alias("month_end"))
               .where((F.col("full_date") >= F.lit(bounds["lo"])) & (F.col("full_date") <= F.lit(bounds["hi"]))))
    spine_count = spine.count()
    series_df = spark.createDataFrame([(series_id,) for series_id in SERIES_MAP], "series_id string")
    grid = spine.crossJoin(series_df)            # date_key, month_end (spine), series_id

    # As-of forward-fill: latest obs whose month_end <= the spine month, per (series, spine month).
    obs_o = obs_last.select(F.col("series_id").alias("o_sid"), F.col("month_end").alias("o_me"), "value")
    asof_w = Window.partitionBy("series_id", "date_key").orderBy(F.col("o_me").desc())
    asof = (grid.join(obs_o, (F.col("series_id") == F.col("o_sid")) & (F.col("o_me") <= F.col("month_end")), "left")
                .withColumn("rn", F.row_number().over(asof_w)).where(F.col("rn") == 1)
                .select("date_key", "series_id", "value"))

    wide = asof.groupBy("date_key").pivot("series_id", list(SERIES_MAP.keys())).agg(F.first("value"))
    staged = wide.select(
        "date_key",
        *[F.col(sid).alias(gold) for sid, gold in SERIES_MAP.items()],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    )
    staged.createOrReplaceTempView("gold_fact_fred_staging")
    step.rows_read = rows_read
    print(f"build_fact_fred: read {rows_read:,} long rows; spine = {spine_count:,} months")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Full rebuild via INSERT OVERWRITE (design §2.2). date_key comes from the dim_date spine, so the
# FK holds by construction; validate post-count == spine month count.
transform_started = datetime.now(timezone.utc)
try:
    spark.sql(f"INSERT OVERWRITE TABLE {TARGET_TABLE} SELECT * FROM gold_fact_fred_staging")

    post_count = spark.table(TARGET_TABLE).count()
    if post_count != spine_count:
        raise AssertionError(f"[{TARGET_TABLE}] Row-count mismatch: spine has {spine_count:,} months, wrote {post_count:,}.")
    step.rows_written = post_count
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=post_count, rows_inserted=post_count, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"build_fact_fred: wrote {post_count:,} monthly rows to {TARGET_TABLE}")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        error_message=f"{type(e).__name__}: {e}", ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise